# 10 · Integration checks: every family in gold

Gold now joins six feature families: calendar (`cal_`), crash history (`crash_`), weather (`wx_`), congestion (`trk_`), taxi trips (`taxi_`) and licensed venues (`lic_`). This notebook checks that the reprocessed feature store has them all, for every hour of the 58 months, how much of each is missing, and that one area-hour in gold equals the values the swimlane notebooks checked by hand.

In [1]:
import json
import os

import pandas as pd
import yaml

# run from the repo root, where main.py, config/ and data/ are
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

with open("config/pipeline.yaml") as f:
    config = yaml.safe_load(f)

# the bulk months only: the replayed hours after go-live are checked in notebook 04
gold = pd.read_parquet("datamart/gold/feature_store", filters=[("date", "<", "2023-01-01")])
gold = gold.drop(columns=["date", "run_id"])
families = config["features"]["families"]
print(f"{len(gold):,} rows; families in config: {families}")

3,265,031 rows; families in config: ['cal', 'crash', 'wx', 'trk', 'taxi', 'lic']


## 1 · Every family, every hour

In [2]:
columns = pd.Series([c for c in gold.columns if c not in ("community_area", "hour_start", "feature_cutoff")])
by_family = columns.groupby(columns.str.split("_").str[0]).apply(list)
for family in families:
    print(f"{family:6} {len(by_family[family]):2} columns: {', '.join(by_family[family])}")
hours = gold["hour_start"].nunique()
print(f"\n{hours:,} hours from {gold['hour_start'].min()} to {gold['hour_start'].max()}; "
      f"77 x {hours:,} = {77 * hours:,} rows expected, {len(gold):,} found; "
      f"duplicate keys: {gold.duplicated(['community_area', 'hour_start']).sum()}")

cal     5 columns: cal_covid_regime, cal_hour, cal_is_holiday, cal_month, cal_weekday
crash   8 columns: crash_area_how_mean_4w, crash_area_notified_1h, crash_area_notified_24h, crash_area_notified_3h, crash_city_how_mean_4w, crash_city_notified_1h, crash_city_notified_24h, crash_city_notified_3h
wx      6 columns: wx_age_hours, wx_dew_point_c, wx_precip_1h_mm, wx_temp_c, wx_visibility_m, wx_wind_ms
trk     2 columns: trk_congestion_index, trk_coverage
taxi    2 columns: taxi_area_trips_1h, taxi_city_trips_1h
lic     2 columns: lic_nightlife_active, lic_package_goods_active

42,403 hours from 2018-03-01 00:00:00 to 2022-12-31 23:00:00; 77 x 42,403 = 3,265,031 rows expected, 3,265,031 found; duplicate keys: 0


## 2 · Missing values, by family and year

In [3]:
year = gold["hour_start"].dt.year
missing = pd.DataFrame({family: gold[by_family[family]].isna().mean(axis=1) for family in families})
(missing.groupby(year).mean() * 100).round(1)

,cal,crash,wx,trk,taxi,lic
hour_start,,,,,,
2018,0.0,2.3,0.1,9.5,0.0,0.0
2019,0.0,0.0,0.2,9.2,0.0,0.0
2020,0.0,0.0,0.2,8.3,0.0,0.0
2021,0.0,0.0,0.0,11.8,0.0,0.0
2022,0.0,0.0,0.4,13.9,0.0,0.0


## 3 · One area-hour, by hand
18:00 on Friday 14 June 2019 in the Loop (area 32). Notebooks 08 and 09 computed these three values straight from silver and bronze: a congestion index of 0.9508, 1,269 taxi trips ended in the hour before, and 282 active nightlife sites. Gold should hold the same numbers.

In [4]:
row = gold[(gold["community_area"] == 32) & (gold["hour_start"] == pd.Timestamp("2019-06-14 18:00"))]
row[["trk_congestion_index", "taxi_area_trips_1h", "lic_nightlife_active"]].T

,869927
trk_congestion_index,0.95083
taxi_area_trips_1h,1269.00000
lic_nightlife_active,282.00000


## 4 · The models trained on every family

In [5]:
with open(os.path.join(config["paths"]["model_bank"], "registry.json")) as f:
    registry = pd.DataFrame(json.load(f))
registry["test_P0"] = registry["test_metrics"].str["P0"]
registry[["model_version", "algorithm", "status", "test_P0"]]

,model_version,algorithm,status,test_P0
0,B0_2021_12_20261016T121530,B0,production,0.290726
1,M1_2021_12_20261016T121530,M1,shadow,0.312129
2,B0_2021_12_20261023T043851,B0,candidate,0.290726
3,M1_2021_12_20261023T043851,M1,candidate,0.310951
